# Physical Chemistry I: Introduction to Gaussian09 via WebMO (Part I)

*Procedure by M. Kubasik (mkubasik@fairfield.edu), 2025*

In this notebook you will work back and forth between two windows:

* **This notebook**, which holds the instructions, the questions you should answer, and a few helper cells.
* **WebMO** at [curie.fairfield.edu](http://curie.fairfield.edu), where you will build molecules and run quantum chemical calculations with Gaussian09.

Blue **Task** boxes are things to do in WebMO. Purple **Question** boxes are things to answer in your notebook based on what you saw.

In [ ]:
#@title Define callout functions (run this cell first)
from IPython.display import HTML, display

_question_index = 0  # Initialize global counter for questions
_indexed_questions = {}  # Stores {original_question_text: assigned_index}
_task_index = 0  # Initialize global counter for tasks
_indexed_tasks = {}  # Stores {original_task_text: assigned_index}

def callout(text, type="task", title=None):
    global _question_index
    global _indexed_questions
    global _task_index
    global _indexed_tasks

    #  This color palette was recommended by Claude to work well
    #  for both light and dark modes.
    colors = {
        "warning":   ("#D97706", "#1C1917", "⚠️ Warning"),
        "important": ("#DC2626", "#FFFFFF", "❕Important ❕"),
        "question":  ("#7C3AED", "#FFFFFF", "❔Question ❔"),
        "note":      ("#15803D", "#FFFFFF", "📘Note "),
        "task":      ("#2563EB", "#FFFFFF", "🎯 Task 🎯"),
        "tip":       ("#0D9488", "#1C1917", "💡Tip"),
    }

    bg, fg, default_label = colors.get(type, colors["task"])
    label = title if title else default_label

    if type == "question":
        original_text = text  # Store the original text before modification
        if original_text in _indexed_questions:
            # If this question has been seen before, use its assigned index
            assigned_idx = _indexed_questions[original_text]
            text = f"Q {assigned_idx}. {original_text}"
        else:
            # If it's a new question, increment and assign a new index
            _question_index += 1
            _indexed_questions[original_text] = _question_index
            text = f"Q {_question_index}. {original_text}"
    elif type == "task":
        original_text = text
        if original_text in _indexed_tasks:
            assigned_idx = _indexed_tasks[original_text]
            text = f"T {assigned_idx}. {original_text}"
        else:
            _task_index += 1
            _indexed_tasks[original_text] = _task_index
            text = f"T {_task_index}. {original_text}"

    display(HTML(f"""
<div style="background:{bg}; color:{fg}; padding:12px 16px;
border-radius:6px; margin:8px 0;">
<strong>{label}</strong>
<br>
<h3 style="color:{fg};">{text}</h3>
</div>
"""))
print("Callouts have been defined.")

> **Tip:** Use *Runtime → Run all* once at the start so that every Task and Question box is displayed and numbered in order.

## 1. Goals

* Learn how to submit a "quantum chemical job" (a request for a quantum chemical calculation) to Gaussian09 via WebMO on University servers.
* Use Gaussian09 to compute the structures of small molecules (formaldehyde, acetaldehyde, and its tautomers) and visualize these structures in WebMO.
* Use WebMO to visualize the molecular orbitals of a small molecule.
* Use Gaussian09 to compute the rotational barrier of ethane.

## 2. Introduction

In today's lab you will learn to use a web-based front end, **WebMO**, to submit quantum chemical calculations to **Gaussian09**. Quantum mechanical calculations are widely used to predict accurate molecular structures, electron distributions, NMR spectra, and infrared spectra.

Increasingly, the results of quantum mechanical calculations are used to interpret experimental results and to guide experimental design for all chemists, not just physical chemists. These computations are used by Professors Kubasik, Miecznikowski, Steffen and others. Gaussian09, developed and published by Gaussian, Inc.<sup>1</sup>, is among the most popular research-grade quantum chemistry packages. You will access Fairfield University's installation of Gaussian09 through the web interface at **curie.fairfield.edu**.

## 3. Procedure

### 3.1 Log in to curie.fairfield.edu

Use a web browser (such as Safari) to go to [https://curie.fairfield.edu](https://curie.fairfield.edu). If the link does not work, you may be off campus; connect to the campus VPN and try again.

| | |
|---|---|
| **Username** | firstname |
| **Password** | firstname |

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/WebMO_Login.png" width="466" alt="WebMO login"></p>

In [ ]:
#@title Task: Log in to WebMO
callout("""Open curie.fairfield.edu in your browser and log in to WebMO with your credentials.""", type="task")

After logging in you should see the **WebMO Job Manager** page:

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.1.3.png" width="483" alt="WebMO Job Manager"></p>

If you do not see this page, let your instructor know. The Job Manager automatically saves all results; you do not have to save any jobs yourself.

In [ ]:
#@title Task: Confirm the Job Manager page
callout("""Confirm that the WebMO Job Manager page is displayed. If it is not, ask your instructor for help.""", type="task")

### 3.2 Calculation of the Structure of Formaldehyde

#### 3.2.1 Review of formaldehyde

Formaldehyde, CH<sub>2</sub>O, is a preservative widely used in biology. You probably met its Lewis structure in General Chemistry. The Lewis structure uses the valence electrons of C, H and O to predict the connectivity of the atoms:

| Lewis structure | VSEPR theory prediction |
|:---:|:---:|
| <img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.1.png" width="191" alt="Lewis"> | <img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.1.1.png" width="239" alt="VSEPR"> |

VSEPR theory predicts that the central carbon is *sp*<sup>2</sup> hybridized and that the bond angles are all approximately 120°:

| Atoms | ∠ |
|:---:|:---:|
| H<sub>left</sub>–C–H<sub>right</sub> | 120° |
| H<sub>left</sub>–C–O | 120° |
| O–C–H<sub>right</sub> | 120° |

These angles are only approximate, and VSEPR theory tells us nothing about bond lengths. We will use Gaussian09 to determine accurate bond angles and bond lengths.

In [ ]:
#@title Task: Set up your formaldehyde results table
callout("""Set up a results table in your notebook like the one below. You will fill in each column as you work through the lab.""", type="task")

| Atoms | measurement | GenChem* | WebMO After Clean-up | QM result |
|:---:|:---:|:---:|:---:|:---:|
| H<sub>left</sub>–C–H<sub>right</sub> | ∠ | | | |
| H<sub>left</sub>–C–O | ∠ | | | |
| O–C–H<sub>right</sub> | ∠ | | | |
| C–H | bond length | | | |
| C=O | bond length | | | |

\* The "GenChem" bond angles are VSEPR predictions. You may use covalent atomic radii to predict bond lengths. (Wikipedia is a good place to find covalent radii, if you don't have them off the top of your heads: https://en.wikipedia.org/wiki/Covalent_radius)

#### 3.2.2 Obtain an initial structure of formaldehyde

In the Job Manager, open the **New Job** menu and select **Create New Job**.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.2.1.png" width="504" alt="Create New Job"></p>

Then use **Lookup → Import by → Name...** to import a set of coordinates for formaldehyde.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.2.2.png" width="743" alt="Lookup by name"></p>

Type `formaldehyde` in the box and click **OK**.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.2.3.png" width="193" alt="Query structure"></p>

In [ ]:
#@title Task: Import formaldehyde by name
callout("""In WebMO, choose New Job → Create New Job, then Lookup → Import by → Name..., and import formaldehyde.""", type="task")

You will now see a representation of formaldehyde in the molecular editor. Is this structure any good? We can check by measuring bond lengths and angles with the **Select** tool (the arrow at left).

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/Build_Molecule_measure.png" width="488" alt="Select tool"></p>

* Select **two** atoms and the bond length is shown at the bottom of the window.
* Select **three** atoms and the bond angle is shown.
* Click on white space to clear your selection.

In [ ]:
#@title Task: Measure the imported structure
callout("""Use the Select tool to measure the three bond angles and the C–H and C=O bond lengths of the molecule WebMO found, and record them.""", type="task")

In [ ]:
#@title Task: Estimate bond lengths from covalent radii
callout("""Look up covalent atomic radii for H, C and O (e.g., https://en.wikipedia.org/wiki/Covalent_radius) and use them to estimate the C–H and C=O bond lengths. Record these, along with the VSEPR angles, in the "GenChem" column of your table.""", type="task")

In [ ]:
#@title Question: Imported angles vs. VSEPR
callout("""How do the bond angles of the imported structure compare to the VSEPR-predicted angles?""", type="question")

In [ ]:
#@title Question: Does VSEPR give bond lengths?
callout("""Does VSEPR theory offer any prediction of bond lengths?""", type="question")

In [ ]:
#@title Question: Imported lengths vs. covalent radii
callout("""How do the bond lengths of the imported structure compare to those estimated from covalent atomic radii?""", type="question")

### 3.3 Clean up the obtained structure

We should "clean up" this structure with a *molecular mechanics* method, because we don't know exactly where WebMO found it. Many imported structures turn out to be highly unrealistic (for example, 3D structures are occasionally fully planar!).

Use the **Cleanup** menu to select either **Comprehensive – Mechanics** or **Comprehensive – Idealized**. Or, the **cleanup** tool is also a good choice (see images below).

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/Build_Molecule.png" width="435" alt="Cleanup menu"> &nbsp; <b>or</b> &nbsp; <img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/cleanup.png" width="284" alt="Cleanup tool"></p>

WebMO will use some model, not fully known to us, to clean up the structure.

In [ ]:
#@title Task: Clean up the structure
callout("""Clean up your formaldehyde structure with Cleanup → Comprehensive – Mechanics, Cleanup → Comprehensive – Idealized, or the cleanup tool.""", type="task")

Measure the cleaned-up structure again with the Select tool:

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/Build_Molecule_measure.png" width="434" alt="Select tool after cleanup"></p>

In [ ]:
#@title Task: Measure the cleaned-up structure
callout("""Measure the new bond lengths and bond angles and record them in the "WebMO After Clean-up" column of your table.""", type="task")

In [ ]:
#@title Question: Effect of cleanup
callout("""How did the bond lengths and bond angles change after cleanup compared with the imported structure?""", type="question")

#### 3.3.1 Select options for submitting a quantum chemical "job"

Next, select one of two University servers to perform the calculation. Use the **Choose engine** link to select "first available", Curie (known as "vml01013"), or "Gibbs".

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/Build_Molecule_choose_engine.png" width="542" alt="Choose engine"></p>

Then use the **Job options** link in the menu at left.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.3.png" width="632" alt="Job options link"></p>

Set the options **as shown below**. Your instructor will explain what a *basis set* means; for now we won't go into detail. Give the job a name that reflects your molecule. The image says "Ethane..."; your window probably shows the molecular formula of formaldehyde.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.4.png" width="455" alt="Configure Gaussian job options"></p>

| Option | Setting |
|---|---|
| Job Name | e.g., `CH2O_B3LYP_Routine` |
| Calculation | Optimize + Vib Freq |
| Theory | B3LYP |
| Basis Set | Basic: 3-21G |
| Charge | 0 |
| Multiplicity | Singlet |

To submit the job, click the **right arrow** at the bottom right:

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.5.png" width="501" alt="Submit arrow"></p>

After you submit, WebMO returns to the Job Manager, which updates your job's status slowly. Reload the page for current status.

In [ ]:
#@title Task: Choose a computational engine
callout("""Click Choose engine and select "first available", Curie (vml01013), or Gibbs.""", type="task")

In [ ]:
#@title Task: Set the Gaussian job options
callout("""Open Job options and set: Calculation = Optimize + Vib Freq, Theory = B3LYP, Basis Set = Basic: 3-21G, Charge = 0, Multiplicity = Singlet. Give the job a descriptive name.""", type="task")

In [ ]:
#@title Task: Submit the formaldehyde job
callout("""Submit the formaldehyde job with the right arrow and monitor its status in the Job Manager (reload the page to update).""", type="task")

#### 3.3.2 Examining your results

When the job status changes from "Queued" or "Running" to "Complete", the job is done. Click the **magnifying glass** icon to view the results.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.6.png" width="823" alt="Completed job"></p>

The page may look as though nothing has changed! Use the Select tool to measure the optimized bond lengths and angles.

In [ ]:
#@title Task: Measure the optimized structure
callout("""Open your completed formaldehyde job (magnifying glass) and measure the optimized bond lengths and bond angles. Record them in the "QM result" column.""", type="task")

In [ ]:
#@title Question: Compare QM results to earlier estimates
callout("""How do the QM-optimized bond angles and bond lengths compare with the VSEPR predictions, the covalent-radii estimates, and the WebMO idealized structure?""", type="question")

Now **scroll down** to the table of numerical results. You will find an entry for **Enthalpy**, given in an energy unit called the **Hartree**. This value is useful *only in comparison to other values calculated under identical computational conditions*, that is, as a Δ*H*.

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.2.7.png" width="294" alt="Results table"></p>

In [ ]:
#@title Task: Record the enthalpy of formaldehyde
callout("""Scroll down to the numerical results and record the enthalpy of formaldehyde in Hartrees.""", type="task")

Your calculation also computed vibrational frequencies and their intensities. The highest-frequency vibrations for this molecule are the C–H stretches (sometimes called "hydride" stretches). Use the **filmstrip** icon near the bottom of the page to animate each vibration.

In [ ]:
#@title Task: Animate the C–H stretches
callout("""Use the filmstrip icon to animate the two highest-energy C–H stretches of formaldehyde, and record the computed IR intensity of each.""", type="task")

In [ ]:
#@title Question: C–H stretching motions
callout("""Describe the motions of the atoms in the two C–H stretches. Does each C–H bond move independently of the other?""", type="question")

In [ ]:
#@title Question: C–H stretch intensities
callout("""What are the computed intensities of the two C–H stretches? Would each appear as an intense or a weak feature in the infrared spectrum?""", type="question")

In [ ]:
#@title Question: Most intense IR feature
callout("""What kind of vibrational motion is predicted to have the most intense feature in the infrared spectrum?""", type="question")

## 4. Acetaldehyde

The first two-carbon aldehyde is acetaldehyde. Repeat the steps above to compute an energy-optimized structure for acetaldehyde.

For this and the following calculations, it is **not** necessary to request vibrational frequencies under Job options (choose **Geometry Optimization** instead of Optimize + Vib Freq).

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.3.1.png" width="352" alt="Acetaldehyde"></p>

In [ ]:
#@title Task: Optimize acetaldehyde
callout("""Build acetaldehyde (Lookup → Import by → Name...), clean it up, and submit a B3LYP/3-21G geometry optimization (no vibrational frequencies).""", type="task")

In [ ]:
#@title Task: Tabulate acetaldehyde results
callout("""When the calculation is done, tabulate your results: the C=O bond length, the H–C–O bond angle, and the enthalpy in Hartrees.""", type="task")

In [ ]:
#@title Question: Acetaldehyde C=O length
callout("""What is the C=O bond length of acetaldehyde?""", type="question")

In [ ]:
#@title Question: Acetaldehyde H–C–O angle
callout("""What is the H–C–O bond angle of acetaldehyde?""", type="question")

In [ ]:
#@title Question: Acetaldehyde vs. formaldehyde
callout("""How do the C=O bond length and H–C–O bond angle of acetaldehyde differ from those of formaldehyde?""", type="question")

In [ ]:
#@title Question: Acetaldehyde enthalpy
callout("""What value did you obtain for the enthalpy of acetaldehyde, in Hartrees?""", type="question")

## 5. Cis-ethenol

Repeat the steps above to compute an energy-optimized structure for **cis-ethenol**. This molecule is a *tautomer* of acetaldehyde (a very specific kind of isomer). Again, vibrational frequencies are not needed.

It has been found (2022) that **Lookup → Import by → Name... → ethenol** loads **trans**-ethenol. You can edit it into the cis structure:

1. Click the **Select** tool arrow.
2. Select the four atoms in the H–O–C–C sequence, in order.
3. Choose **Dihedral Angle** under the **Adjust** menu.
4. Enter `0`.

As before, clean up the molecule with either "Mechanics" or "Idealized".

In [ ]:
#@title Task: Build cis-ethenol
callout("""Import ethenol by name, then convert it to cis-ethenol by setting the H–O–C–C dihedral angle to 0 (Select tool → select four atoms → Adjust → Dihedral Angle).""", type="task")

In [ ]:
#@title Task: Optimize cis-ethenol
callout("""Clean up cis-ethenol, submit a B3LYP/3-21G geometry optimization (no vibrational frequencies), and record its enthalpy in Hartrees.""", type="task")

Use the enthalpy of cis-ethenol and the value you obtained for acetaldehyde to calculate the enthalpy change for the reaction:

<p align="center"><img src="https://raw.githubusercontent.com/mkubasik/pchem1LabNotebooks/main/GaussianI/img/3.3.2.png" width="395" alt="Keto-enol tautomerization"></p>

$$\Delta H_{\text{reaction}} = H_{\text{products}} - H_{\text{reactants}}$$

Reactions with large, negative enthalpy changes are favored. Which form of C<sub>2</sub>H<sub>4</sub>O is favored, according to this (gas-phase) calculation?

You can use the helper cell below to convert your Δ*H* from Hartrees to kJ/mol and kcal/mol.

In [ ]:
#@title ΔH calculator (Hartree → kJ/mol, kcal/mol)
#@markdown Enter the enthalpies (in Hartrees) reported by WebMO, then run the cell.
H_reactant = 0.0  #@param {type:"number"}
H_product = 0.0  #@param {type:"number"}

HARTREE_TO_KJ = 2625.5
HARTREE_TO_KCAL = 627.5

dH = H_product - H_reactant
print(f"ΔH = {dH:.6f} Hartree")
print(f"   = {dH*HARTREE_TO_KJ:.2f} kJ/mol")
print(f"   = {dH*HARTREE_TO_KCAL:.2f} kcal/mol")
if dH < 0:
    print("ΔH < 0: the product is lower in enthalpy (conversion favored by enthalpy).")
elif dH > 0:
    print("ΔH > 0: the reactant is lower in enthalpy (conversion disfavored by enthalpy).")

In [ ]:
#@title Question: Cis-ethenol enthalpy
callout("""What is the enthalpy reported for cis-ethenol, this tautomer of acetaldehyde?""", type="question")

In [ ]:
#@title Question: ΔH acetaldehyde → cis-ethenol
callout("""Report the ΔH for the conversion of acetaldehyde to cis-ethenol.""", type="question")

In [ ]:
#@title Question: Is cis-ethenol formation favored?
callout("""Is the conversion of acetaldehyde to cis-ethenol favored, according to your calculation?""", type="question")

## 6. Trans-ethenol

Repeat the steps above to compute an energy-optimized structure for **trans-ethenol**, another tautomer of acetaldehyde. Again, vibrational frequencies are not needed. (Importing "ethenol" by name gives the trans form directly.)

In [ ]:
#@title Task: Optimize trans-ethenol
callout("""Import trans-ethenol by name, clean it up, submit a B3LYP/3-21G geometry optimization (no vibrational frequencies), and record its enthalpy in Hartrees.""", type="task")

In [ ]:
#@title Question: Trans-ethenol enthalpy
callout("""What is the enthalpy reported for trans-ethenol, this tautomer of acetaldehyde?""", type="question")

In [ ]:
#@title Question: ΔH acetaldehyde → trans-ethenol
callout("""Report the ΔH for the conversion of acetaldehyde to trans-ethenol.""", type="question")

In [ ]:
#@title Question: Is trans-ethenol formation favored?
callout("""Is the conversion of acetaldehyde to trans-ethenol favored, according to your calculation?""", type="question")

## References

1. Frisch, M. J.; Trucks, G. W.; Schlegel, H. B.; Scuseria, G. E.; Robb, M. A.; Cheeseman, J. R.; Scalmani, G.; Barone, V.; Mennucci, B.; Petersson, G. A.; Nakatsuji, H.; Caricato, M.; Li, X.; Hratchian, H. P.; Izmaylov, A. F.; Bloino, J.; Zheng, G.; Sonnenberg, J. L.; Hada, M.; Ehara, M.; Toyota, K.; Fukuda, R.; Hasegawa, J.; Ishida, M.; Nakajima, T.; Honda, Y.; Kitao, O.; Nakai, H.; Vreven, T.; Montgomery, J. A., Jr.; Peralta, J. E.; Ogliaro, F.; Bearpark, M.; Heyd, J. J.; Brothers, E.; Kudin, K. N.; Staroverov, V. N.; Keith, T.; Kobayashi, R.; Normand, J.; Raghavachari, K.; Rendell, A.; Burant, J. C.; Iyengar, S. S.; Tomasi, J.; Cossi, M.; Rega, N.; Millam, J. M.; Klene, M.; Knox, J. E.; Cross, J. B.; Bakken, V.; Adamo, C.; Jaramillo, J.; Gomperts, R.; Stratmann, R. E.; Yazyev, O.; Austin, A. J.; Cammi, R.; Pomelli, C.; Ochterski, J. W.; Martin, R. L.; Morokuma, K.; Zakrzewski, V. G.; Voth, G. A.; Salvador, P.; Dannenberg, J. J.; Dapprich, S.; Daniels, A. D.; Farkas, O.; Foresman, J. B.; Ortiz, J. V.; Cioslowski, J.; Fox, D. J. *Gaussian 09*, Rev. B.01; Gaussian, Inc.: Wallingford, CT, 2010.
2. https://en.wikipedia.org/wiki/Paraldehyde (accessed September 13, 2022)

## 7. Appendix

### 7.1 Energy units

1 Hartree = 2625.5 kJ/mol = 627.5 kcal/mol = 27.211 eV = 219474.6 cm<sup>−1</sup>

### 7.2 Literature data

Good web-based sources for literature data:

* [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/)
* Wikipedia

### 7.3 Compounds

| Compound | CAS Registry Number |
|---|---|
| Formaldehyde | 50-00-0 |
| Acetaldehyde | 75-07-0 |
| Ethenol | 557-75-5 |